In [52]:
from utils.orthanc_utils import *
from utils.db_utils import *
from utils.plot_utils import *
from utils.mri_sorter import MRI_Sorter
from utils.model_utils import *
from scipy.ndimage import zoom

def get_volumes(dicoms):    
    df = pd.DataFrame(
        [{"SliceLocation": d.SliceLocation, "InstanceNumber": d.InstanceNumber, "PixelArray": d.pixel_array} for d in dicoms]
    ).sort_values(["SliceLocation", "InstanceNumber"])

    d0 = dicoms[0]
    pixel_spacing = float(d0.PixelSpacing[0])
    thickness = float(getattr(d0, "SpacingBetweenSlices", getattr(d0, "SliceThickness", np.nan)))
    voxel_size = (pixel_spacing ** 2 * thickness) / 1000
    
    array_4d = []
    for _, slice_df in df.groupby("SliceLocation"):
        time_array = []
        for _, time_df in slice_df.groupby("InstanceNumber"):
            pixel_array = time_df["PixelArray"].iloc[0]
            time_array.append(pixel_array)

        slice_array = np.stack(time_array, axis=-1)
        array_4d.append(slice_array)

    array_4d = np.stack(array_4d, axis=-2)

    labels = {500: "lv", 1000: "rv", 1500: "lv_myo", 2000: "rv_myo"}
    curves = {
        name: np.sum(array_4d == val, axis=(0, 1, 2)) * voxel_size
        for val, name in labels.items()
    }
    lv, rv = curves["lv"], curves["rv"]
    lv_edv, lv_esv = lv.max(), lv.min()
    rv_edv, rv_esv = rv.max(), rv.min()

    metrics = {
        "lv_edv": lv_edv,
        "lv_esv": lv_esv,
        "lv_ef": (lv_edv - lv_esv) / lv_edv,
        "lv_mass": np.mean(curves["lv_myo"]) * 1.05,
        "rv_edv": rv_edv,
        "rv_esv": rv_esv,
        "rv_ef": (rv_edv - rv_esv) / rv_edv,
        "rv_mass": np.mean(curves["rv_myo"]) * 1.05,
    }

    return metrics


In [53]:

METRICS_PATH = 'clasp_metrics.csv'
db = TinyDB('image_clasp_db.json')
metrics_df = pd.read_csv(METRICS_PATH)


In [54]:
db_studies = fetch_db_studies()


In [55]:
db = TinyDB(DB_PATH)
db_studies = fetch_db_studies()
orthanc_studies = fetch_orthanc_studies()

studies = {db_study['orthanc_study_id']:Study.from_dict(db_study) for db_study in db_studies}
existing_study_ids = studies.keys()

for orthanc_study_info in orthanc_studies:
    if orthanc_study_info["ID"] in existing_study_ids:
        study = studies[orthanc_study_info["ID"]]
    else:
        study = Study(orthanc_study_info)
        
    # mri_sorter = MRI_Sorter(study)
    # series_type_df = mri_sorter.sort_df
    series_list = fetch_orthanc_series_for_study(study.orthanc_study_id)

    for series_info in series_list:
        series = Series(series_info)
            
        if series.dl_orthanc_id and not series.roundel_orthanc_id: 
            print(1)

In [56]:
fetch_db_series(study, series['ID'])

TypeError: 'Series' object is not subscriptable

In [38]:
series_info['ID']

'ffdd7f43-a6e1baa2-05fe4714-f1773c5b-540487f2'

[{'orthanc_series_id': 'ffdd7f43-a6e1baa2-05fe4714-f1773c5b-540487f2',
  'series_uid': '1.2.826.0.1.3680043.8.498.31417026834048461663089161915243962494',
  'series_description': 'SAX DL Segmented',
  'series_type': 'Cine Stack',
  'series_orientation': 'SAX',
  'dl_orthanc_id': None,
  'roundel_orthanc_id': None},
 {'orthanc_series_id': 'ffdd7f43-a6e1baa2-05fe4714-f1773c5b-540487f2',
  'series_uid': '1.2.826.0.1.3680043.8.498.31417026834048461663089161915243962494',
  'series_description': 'SAX DL Segmented',
  'series_type': 'Cine Stack',
  'series_orientation': 'SAX',
  'dl_orthanc_id': None,
  'roundel_orthanc_id': None}]

In [36]:
study.series_list[series_info['ID']]

TypeError: list indices must be integers or slices, not str

In [33]:
series

In [32]:
series.__dict__

{'orthanc_series_id': 'ffdd7f43-a6e1baa2-05fe4714-f1773c5b-540487f2',
 'series_uid': '1.2.826.0.1.3680043.8.498.31417026834048461663089161915243962494',
 'series_description': 'SAX DL Segmented',
 'series_type': None,
 'series_orientation': None,
 'dl_orthanc_id': None,
 'roundel_orthanc_id': None}

In [25]:
series_list

[]

In [24]:
series_list

[]

In [7]:
series_list

NameError: name 'series_list' is not defined

In [ ]:

# rows = []
# for study in db.search(StudyQuery.series.any(SeriesQuery.dl_orthanc_id != None)):
    # dl_processed_series_list = [series for series in study.get("series", []) if series.get("dl_orthanc_id") != None]

#     image_dicoms = [d for s in dl_processed_series_list for d in fetch_dicoms_for_series(s['orthanc_series_id'])]
#     mask_dicoms = [d for s in dl_processed_series_list for d in fetch_dicoms_for_series(s['dl_orthanc_id'])]
#     metrics = get_volumes(mask_dicoms)
#     metrics['orthanc_study_id'] = study['orthanc_study_id']
#     rows.append(metrics)

#     masked_images = [image.pixel_array * (mask.pixel_array > 0) for image, mask in zip(image_dicoms, mask_dicoms) ]
#     new_orthanc_id = send_series_to_orthanc(masked_images, image_dicoms, new_description='Roundel Processed')

# metrics_df = pd.concat([metrics_df, pd.DataFrame(rows)], ignore_index=True).set_index('orthanc_study_id')
# metrics_df.to_csv(METRICS_PATH)


In [20]:
study

{'orthanc_study_id': 'f8c6d5e6-cb6ccdff-569085d3-b2a6ca35-bfa56a60',
 'study_uid': '1.3.12.2.1107.5.2.18.42363.30000026032008405178000000016',
 'patient_name': 'pat_01',
 'patient_id': 'JAC',
 'patient_sex': 'M',
 'patient_dob': '19920101',
 'patient_age': 34,
 'study_date': '20260320',
 'series': [{'orthanc_series_id': '0254aa4e-bc4024c8-91a1f641-1d0b35bc-14f70c08',
   'series_uid': '1.3.12.2.1107.5.2.18.42363.2026032015192880208240051.0.0.0',
   'series_description': 'post_MOLLI_4s(1s)3s(1s)2s_256 sax b_MOCO_SPSIR',
   'series_type': 'Single-Short Multi-Heartbeat',
   'series_orientation': None,
   'dl_orthanc_id': None,
   'roundel_orthanc_id': None},
  {'orthanc_series_id': '02f8212f-717cff89-a7569770-edcfa636-62f5b8fc',
   'series_uid': '1.3.12.2.1107.5.2.18.42363.2026032014534510536237854.0.0.0',
   'series_description': 'SSFP_atria stack',
   'series_type': 'Cine Stack',
   'series_orientation': None,
   'dl_orthanc_id': None,
   'roundel_orthanc_id': None},
  {'orthanc_series_i

In [19]:
dl_processed_series_list[4]

{'orthanc_series_id': '51806895-7392a944-a06def7e-a6edf4b4-fb230242',
 'series_uid': '1.3.12.2.1107.5.2.18.42363.2026032014481199798936861.6.0.0',
 'series_description': 'SSFP_SAX_10sl each slice Higher TR',
 'series_type': 'Cine Stack',
 'series_orientation': 'SAX',
 'dl_orthanc_id': '142c4033-33b18c20-bf0fb123-5a79a1cb-69f18377',
 'roundel_orthanc_id': None}